In [2]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from typing import Tuple, Optional


def load_image(image_path: str) -> np.ndarray:
    """Загрузка изображения и преобразование в grayscale numpy array"""
    img = Image.open(image_path).convert("L")
    return np.array(img)


def normalize_image(image: np.ndarray) -> np.ndarray:
    """Нормализация изображения"""
    mean = np.mean(image)
    std = np.std(image)
    return (image - mean) / (std + 1e-10)


def correlation_2d(image: np.ndarray, template: np.ndarray) -> np.ndarray:
    """
    Вычисление корреляции между изображением и шаблоном
    Возвращает карту корреляции
    """
    # Нормализуем оба изображения
    image_norm = normalize_image(image)
    template_norm = normalize_image(template)

    # Получаем размеры
    img_h, img_w = image_norm.shape
    tpl_h, tpl_w = template_norm.shape

    # Инициализируем карту корреляции
    corr_map = np.zeros((img_h - tpl_h + 1, img_w - tpl_w + 1))

    # Вычисляем корреляцию для каждого возможного положения шаблона
    for y in range(corr_map.shape[0]):
        for x in range(corr_map.shape[1]):
            # Вырезаем окно из изображения
            window = image_norm[y : y + tpl_h, x : x + tpl_w]
            # Вычисляем корреляцию
            corr = np.sum(window * template_norm)
            corr_map[y, x] = corr

    return corr_map


def find_max_correlation(corr_map: np.ndarray) -> Tuple[Tuple[int, int], float]:
    """Нахождение координат максимальной корреляции"""
    max_val = np.max(corr_map)
    max_pos = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    return max_pos, max_val


def save_color_correlation_map(corr_map: np.ndarray, output_path: str) -> None:
    """Сохраняет карту корреляций"""
    plt.figure(figsize=(10, 8))
    plt.imshow(corr_map, cmap="jet")
    plt.colorbar()
    plt.title("Correlation Map")
    plt.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close()


def template_matching(
    image_path: str, template_path: str, output_color_path: Optional[str] = None
) -> Tuple[Tuple[int, int], float, np.ndarray]:
    """Основная функция для поиска шаблона на изображении"""
    # Загрузка изображений
    image = load_image(image_path)
    template = load_image(template_path)

    # Вычисление корреляции
    corr_map = correlation_2d(image, template)

    # Сохранение карты корреляций
    if output_color_path:
        save_color_correlation_map(corr_map, output_color_path)

    # Нахождение максимума
    (y, x), max_val = find_max_correlation(corr_map)

    return (x, y), max_val, corr_map

In [3]:
# Параметры
image_path = "origins/novak.jpg"
template_path = "origins/eye.jpg"

position, correlation, corr_map = template_matching(
    image_path,
    template_path,
    output_color_path="results/task_3/correlation_map_2.png",
)

print(f"Шаблон найден в позиции: {position} с корреляцией: {correlation:.4f}")

Шаблон найден в позиции: (np.int64(78), np.int64(75)) с корреляцией: 1964.3514
